## PHASE 2: Clean and Prepare Samples

#### Objective
- A cleaning strategy for the Methodology chapter
- Clean text in a consistent format for both corpora
- For earnings calls: split into prepared remarks vs Q&A (using [Operator Instructions] boundary)
- For EDGAR: filtered to substantive sections only (1, 1A, 7, 8)
- Each "document unit" tagged with metadata so the results can be traced back to source





Taking the raw edgar_sample_100.parquet and earnings_sample_100.parquet, apply targeted cleaning based on what was discovered in Phase 1, and produce two clean parquet files ready for chunking in Phase 3.
By the end of Phase 2, it should produce:

For EDGAR:

- Index sections 1, 1A, 7, 8 only. These are the analytical core.
- Treat each section as its own retrievable unit (a "document" = one section of one filing).
- Light cleaning only: collapse whitespace, strip non-printable chars. Do not strip boilerplate - 10-K language is the content.
- Skip filings where all four target sections are missing or empty.

For earnings transcripts:

- Split each transcript on [Operator Instructions] into two parts: prepared remarks and Q&A.
- Treat each part as a separate document unit (so 100 transcripts → up to 200 units).
- For transcripts without [Operator Instructions], keep as one undifferentiated unit, flagged as such.
- Light cleaning only: normalize whitespace, fix spacing around speaker colons.
- Not stripping disclaimers :- **Justification:** it introduces a confound. We want to compare retrievers, not cleaning strategies.

#### Output schema (unified across both corpora):             
<pre>
Field               Description
doc_id              unique ID like edgar_0001_section_7 or earnings_0042_qa
source              "edgar" or "earnings"
corpus_type         "formal" or "conversational"
subtype             EDGAR section name OR "prepared_remarks"/"qa"/"full"
text                the cleaned text content
char_length         length of cleaned text
metadata            dict of extra info (filename, ticker, date, etc.)

This unified schema is what makes downstream code clean — retrievers don't care if it's EDGAR or earnings, they just see text and doc_id.

cleaner.py module:

- basic_clean - minimal text normalization shared by both corpora. We avoid heavy cleaning intentionally.
- prepare_edgar_documents - explodes wide EDGAR DataFrame (1 row × 23 section columns) into long format (4 rows per filing, one per substantive section). Drops empty/stub sections.
- split_transcript_on_qa - uses the [Operator Instructions] marker you validated in Phase 1 to split each call into prepared remarks and Q&A.
- prepare_earnings_documents - produces two units per call (or one if no boundary detected).
- combine_corpora - utility for later phases when you want a single unified corpus.

Why this design:

Each text unit gets a unique doc_id and rich metadata. When you later run retrieval and a chunk is returned, you can trace it back to: 
which corpus, which company, which section, which call. This is critical for your Results chapter — you can't analyze where retrievers 
succeed or fail without that traceability.

## Couple of queries I have on the phase 2 code and details you shared:
- Can I use the same kernel "thesis (Python 3.13.0)" that I used in Phase 1 to keep same standard for the entire thesis work?
- Index sections 1, 1A, 7, 8 only. These are the analytical core. But you did mention 2 other sections that are important, than why not add section 15 as it is content rich and section 3 which is legal"
- What is the meaning of "Treat each section as its own retrievable unit (a "document" = one section of one filing)."? Does it mean one section for a company is one document which can be one chunk? Is it something that can be done like chunking each section in a separate single unit for a company?
- Why "Light cleaning only: collapse whitespace, strip non-printable chars. Do not strip boilerplate - 10-K language is the content." because 10K content is critical and cannot be stripped away which can cause stripping of valuable information?
For earning
- So when you say "Treat each part as a separate document unit (so 100 transcripts → up to 200 units)." do you create this separation while chunking, like add metadata or something?
- Is the "Output schema (unified across both corpora)", is that for the chunks as to my knowledge that is the actual place where you create smaller unit from a large document? and also in this unified schema's metadata, is it where we are going to add the separation for earning transcripts like one prepared remarks and Q&A
- In the function "def prepare_edgar_documents" can you explain the entire function in simple terms and about 'doc_id = f"edgar_{filing_idx:04d}_{section}'. And what do you mean by " explodes wide EDGAR DataFrame (1 row × 23 section columns) into long format (4 rows per filing, one per substantive section)." Does this function creates 4 rows for a single 10k filling of a company for each section we finalized like company x section 1 is row 1, same company x section 1A is row 2, company x section 7 row 3 and same company x section 8 is row 4, also what is the benefit of creating 4 rows per section for single company filling? Also in the same fUnction why set the filename like "filename = row.get("filename", f"unknown_{filing_idx}")" can you show me an example of how the output will look like for this line?
- Can you talk about the function "def split_transcript_on_qa(text: str) -> tuple[str, str, bool]:" and explain in simple terms maybe with an example would be nice. Also explain samething here "earnings_{call_idx:04d}_qa" what output this makes what is this :04d and will be the output from {call_idx:04d}. Further if metadata is for both halves what is 
```
if len(prepared) >= 500:
                rows.append({
                    "doc_id": f"earnings_{call_idx:04d}_prepared",
                    "source": "earnings",
                    "corpus_type": "conversational",
                    "subtype": "prepared_remarks",
                    "text": prepared,
                    "char_length": len(prepared),
                    "metadata": meta,
                })
```
is this per column data in the dataframe as I could see the funcction "prepare_earnings_documents" spits out a pd.Dataframe
- What is the purpose of using the line "sys.path.insert(0, str(PROJECT_ROOT))" underneath the code "PROJECT_ROOT = Path(__file__).resolve().parent.parent"?
- Why use below lines in the setup cell block
```
%load_ext autoreload
%autoreload 2"
```
- What do you mean when you say "What you should see: roughly ~85 × 4 = 340 document units, but actual count depends on how many filings have substantive content in each section. Some filings might lose 1–2 sections that were too short." under "Process EDGAR:"? I think my understanding is close when I said 4 rows per company's filling but why 85 and I think x 4 is for 4 target sections but still want to know how the pd dataframe will look like since I have not ran the code blocks, just writing all my queries before running code
- When you are suggesting to run the "Combine and save:" block it is saving files :
```
data/processed/edgar_processed.parquet exists
data/processed/earnings_processed.parquet exists
data/processed/combined_corpus.parquet exists
 ```
 Isn't that troublesome since this is just a sample run with 100 files what if I run it across full dataset woudn't that override the sample files.
- Can you show me one sample output of "summary" for cell "Cell 8 — Save processing summary for your notes:" and one for "phase_02_processing_summary.json"









- Not finally overall, tell me is this a right approach to write my queries one by one instead of asking the question every time I feel I don't know something from the code you gave me to make the most of my time and get to the end results fast as majority of time is something will be occupied by the writing part for the paper. Also one of my other anxiety point is since we are only working with 100 samples how or what is the stratergy to cover full dataset. Earnings may be all are in single dataset but regulatory fillings are per year basis?

In [1]:
import sys
from pathlib import Path

# Path.cwd().parent
sys.path.insert(0, str(Path.cwd().parent))

### 1 : Setup

In [2]:
%load_ext autoreload
%autoreload 2
# %reload_ext autoreload # when needed to reload
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import json

from src.preprocessing.cleaners import (
    prepare_edgar_documents,
    prepare_earnings_documents,
    combine_corpora,
)

SAMPLES_DIR = PROJECT_ROOT / "data" / "samples"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
NOTES_DIR = PROJECT_ROOT / "notes"

print(f"Loading from: {SAMPLES_DIR}")
print(f"Saving to: {PROCESSED_DIR}")

Loading from: d:\General IT\AI-ML-LJMU\final_thesis\code\data\samples
Saving to: d:\General IT\AI-ML-LJMU\final_thesis\code\data\processed


<h4>sys.path.insert(0, str(PROJECT_ROOT)) — what it does</h4>
<hr>
<pre>
- Python only looks for modules in directories listed in sys.path. By default, that's a few standard locations 
plus the directory the script lives in.
- My src/ folder isn't automatically searchable. So when one write from src.data_loading.loaders import ..., 
Python can't find src unless one tell it where to look.
- sys.path.insert(0, str(PROJECT_ROOT)) says: "Add the project root to the front of the module search path." 
Now src/ is findable from anywhere.
- The 0 means "insert at position 0" — i.e., search this path first, before built-in paths. 
This protects against name collisions (in case you accidentally created a src module name that conflicts with something else).
- This pattern is necessary for notebooks living in notebooks/ that need to import from src/. 
Without it, you'd get ModuleNotFoundError.
</pre>
<br>
<pre>%load_ext autoreload
%autoreload 2
</pre>

This is exactly the bug that bit me earlier with inspect_dataframe.

- **%load_ext autoreload -** load the autoreload extension
- **%autoreload 2 - set mode 2:** automatically reload all imported modules before executing each cell
- With this active, I can edit loaders.py, re-run a notebook cell and the new code runs immediately. 
No kernel restart needed.
- I should put this at the top of every notebook from now on.

### 2 : Load the Phase 1 outputs:

In [3]:
edgar_raw = pd.read_parquet(SAMPLES_DIR / "edgar_sample_100.parquet")
earnings_raw = pd.read_parquet(SAMPLES_DIR / "earnings_sample_100.parquet")

print(f"Loaded {len(edgar_raw)} EDGAR filings")
print(f"Loaded {len(earnings_raw)} earnings transcripts")

Loaded 100 EDGAR filings
Loaded 100 earnings transcripts


### 3 : Process EDGAR:

In [4]:
# Should see: roughly ~85 × 4 = 340 document units, but actual count depends on how many filings have 
# substantive content in each section. Some filings might lose 1–2 sections that were too short.

edgar_processed = prepare_edgar_documents(edgar_raw)

print(f"EDGAR: {len(edgar_raw)} filings → {len(edgar_processed)} document units")
print("\nUnits per section:")
print(edgar_processed["subtype"].value_counts())
print("\nLength stats per section:")
print(edgar_processed.groupby("subtype")["char_length"].describe().round(0))

EDGAR: 100 filings → 319 document units

Units per section:
subtype
section_1     88
section_7     88
section_1A    84
section_8     59
Name: count, dtype: int64

Length stats per section:
            count     mean      std     min      25%      50%       75%  \
subtype                                                                   
section_1    88.0  48857.0  42757.0  1509.0  20322.0  33852.0   61995.0   
section_1A   84.0  87837.0  61849.0  1163.0  44500.0  67010.0  124660.0   
section_7    88.0  47193.0  29570.0  5576.0  23260.0  42831.0   62638.0   
section_8    59.0  95596.0  63566.0   520.0  54835.0  93983.0  122936.0   

                 max  
subtype               
section_1   195132.0  
section_1A  270359.0  
section_7   121527.0  
section_8   420160.0  


### How to Read Quartiles and "Bimodal"

A boxplot or describe() output gives you five numbers that summarize a distribution:

- min: smallest value
- 25th percentile (Q1): 25% of values fall below this
- 50th percentile / median (Q2): half the values are below, half above
- 75th percentile (Q3): 75% of values fall below this
- max: largest value

Applied to your section_1A data:

- 25% of Risk Factors sections are shorter than 44,500 chars
- 50% are shorter than 67,010 chars (median)
- 75% are shorter than 124,660 chars
- 25% are between 67K and 125K chars

The gap between Q1 and Q3 (the "interquartile range") tells how spread out the middle 50% is. For section_1A, that's 44K to 125K - huge variance. Some companies write minimal risk factors; others write enormous ones.
On "bimodal": a distribution is bimodal when it has two peaks instead of one. Looking at section_8 again - min 520, but Q1 already 54,835 - there's a giant jump from "barely above threshold" to "tens of thousands of chars." This suggests two distinct groups: (a) substantive financial statements, and (b) short pointer/reference sections. 
They cluster around two values rather than one. "Bimodal-looking" would be more accurate without a histogram to confirm.

### 4 : Spot-check one EDGAR unit:

In [5]:
sample = edgar_processed.iloc[0]
print(f"doc_id: {sample['doc_id']}")
print(f"subtype: {sample['subtype']}")
print(f"length: {sample['char_length']:,} chars")
print(f"metadata: {sample['metadata']}")
print(f"\nFirst 800 chars of cleaned text:\n")
print(sample["text"][:800])

doc_id: edgar_0000_section_1
subtype: section_1
length: 113,938 chars
metadata: {'filename': '103682_2020.htm', 'cik': '103682', 'year': '2020', 'filing_idx': 0}

First 800 chars of cleaned text:

Item 1. Business GENERAL Dominion Energy, headquartered in Richmond, Virginia and incorporated in Virginia in 1983, is one of the nation’s largest producers and distributors of energy. Dominion Energy’s strategy is to be a leading sustainable provider of electricity, natural gas and related services to customers primarily in the eastern and Rocky Mountain regions of the U.S. As of December 31, 2020, Dominion Energy’s portfolio of assets includes approximately 30.2 GW of electric generating capacity, 10,500 miles of electric transmission lines, 85,600 miles of electric distribution lines and 94,200 miles of gas distribution mains and related service facilities, which are supported by 6,200 miles of gas transmission, gathering and storage pipeline. In addition, Dominion Energy owns approxima


### 5 : Process earnings:

In [6]:
# should see: Roughly 77 calls split into 2 units each (~154 units) + ~23 calls as single 'full' units = ~177 total. 
# Exact numbers depend on which 77/100 had the boundary marker.

earnings_processed = prepare_earnings_documents(earnings_raw)

print(f"Earnings: {len(earnings_raw)} calls → {len(earnings_processed)} document units")
print("\nUnits by subtype:")
print(earnings_processed["subtype"].value_counts())

# Of original 100 calls, how many were successfully split?
split_count = sum(
    earnings_processed["metadata"].apply(lambda m: m.get("was_split", False))
)
unique_calls_split = earnings_processed[
    earnings_processed["metadata"].apply(lambda m: m.get("was_split", False))
]["metadata"].apply(lambda m: m["call_idx"]).nunique()

print(f"\nCalls successfully split into prepared+QA: {unique_calls_split}")
print(f"Calls kept as 'full' (no boundary detected): "
      f"{(earnings_processed['subtype'] == 'full').sum()}")

Earnings: 100 calls → 160 document units

Units by subtype:
subtype
prepared_remarks    60
qa                  60
full                40
Name: count, dtype: int64

Calls successfully split into prepared+QA: 73
Calls kept as 'full' (no boundary detected): 40


<h4> ** First run ** </h4>
Out of 100 transcripts:

74 had a detectable Q&A boundary → each was split into a prepared_remarks unit AND a qa unit
26 had no detectable Q&A boundary → kept whole as "full" units

The anomaly is here: If 74 transcripts were split, we should get 74 prepared_remarks units AND 74 qa units (one of each per split call). The Phase 2 output shows:

- 74 qa units ✓ (matches expectation)
- 23 prepared_remarks units ✗ (should be 74, but is 23)
- 26 full units ✓ (matches expectation)

So 51 prepared_remarks units went missing. Where did they go?
- Likely cause: In prepare_earnings_documents, there's a length filter, units shorter than 500 characters get dropped before being added to the output. The code looks like this:
<pre>
if len(prepared) >= 500:
    rows.append({...prepared_remarks row...})
if len(qa) >= 500:
    rows.append({...qa row...})
</pre>

<hr>

<h4> ** Second run post applying fix** </h4>

Out of 100 transcripts:

60 had a detectable Q&A boundary → each was split into a prepared_remarks unit AND a qa unit
40 had no detectable Q&A boundary → kept whole as "full" units

The anomaly is here: If 74 transcripts were split, we should get 74 prepared_remarks units AND 74 qa units (one of each per split call). The Phase 2 output shows:

- 60 qa units ✓ (matches expectation)
- 60 prepared_remarks units ✓ (matches expectation)
- 40 full units ✓ (matches expectation)

<hr>

In [7]:
# How many transcripts had a participant header stripped?
header_stripped_count = sum(
    1 for _, row in earnings_processed.iterrows()
    if row["metadata"].get("header_stripped", False)
)
# Note: each call produces up to 2 rows (prepared + qa), so count unique calls
unique_stripped = earnings_processed[
    earnings_processed["metadata"].apply(lambda m: m.get("header_stripped", False))
]["metadata"].apply(lambda m: m["call_idx"]).nunique()

print(f"Calls with stripped participant header: {unique_stripped}")
# Expected: 32, matching your Phase 1 finding

# Sanity: confirm no 'Executives:' remains at the start of any cleaned transcript
starts_with_executives = sum(
    1 for text in earnings_processed["text"]
    if text.strip().lower().startswith("executive")
)
print(f"Cleaned units still starting with 'Executive': {starts_with_executives}")
# Expected: 0

Calls with stripped participant header: 32
Cleaned units still starting with 'Executive': 0


### 6 : Spot-check earnings split quality:

- To verify: prepared remarks should end before any Q&A starts, and the Q&A section should start with [Operator Instructions] followed by analyst questions. 
- If a prepared section ends with a question or a Q&A section starts mid-management-statement, this means the boundary detection has issues.

In [8]:
# Pick a call that was split and inspect both halves
split_units = earnings_processed[earnings_processed["subtype"] != "full"]
sample_call_idx = split_units.iloc[0]["metadata"]["call_idx"]

call_units = earnings_processed[
    earnings_processed["metadata"].apply(lambda m: m["call_idx"] == sample_call_idx)
]

for _, unit in call_units.iterrows():
    print(f"=== {unit['doc_id']} ({unit['subtype']}, {unit['char_length']:,} chars) ===")
    print(f"Company: {unit['metadata']['company']}")
    print(f"\nFirst 400 chars:")
    print(unit["text"][:400])
    print(f"\nLast 200 chars:")
    print(unit["text"][-200:])
    print("\n" + "="*80 + "\n")

=== earnings_0000_prepared (prepared_remarks, 11,257 chars) ===
Company: Huntington Ingalls Industries

First 400 chars:
Operator : Good day, ladies and gentlemen, and welcome to the Huntington Ingalls Industries Second Quarter 2016 Earnings Call. [Operator Instructions] As a reminder, this call is being recorded. I would now like to introduce your host for today's conference, Mr. Dwayne Blake, Vice President of Investor Relations. Sir, you may begin. Dwayne Blake : Thanks, Kyla. Good morning and welcome to the Hunt

Last 200 chars:
e on the call, please limit yourself to one initial question and one follow-up so we can get as many people through the queue as possible. Kyla, I'll turn it over to you to manage the Q&A. Operator : 


=== earnings_0000_qa (qa, 35,876 chars) ===
Company: Huntington Ingalls Industries

First 400 chars:
[Operator Instructions] And our first question comes from the line of Gautam Khanna of Cowen and Company. Gautam Khanna : I was wondering if you could quanti

### 7 : Combine and save:

In [11]:
combined = combine_corpora(edgar_processed, earnings_processed)

print(f"Combined corpus: {len(combined)} total document units")
print(f"\nBy source:")
print(combined["source"].value_counts())
print(f"\nBy corpus_type:")
print(combined["corpus_type"].value_counts())
print(f"\nLength distribution by source:")
print(combined.groupby("source")["char_length"].describe().round(0))

# Tag filenames with sample size to prevent overwrites
edgar_size = len(edgar_raw)
earnings_size = len(earnings_raw)

edgar_out = PROCESSED_DIR / f"edgar_processed_n{edgar_size}.parquet"
earnings_out = PROCESSED_DIR / f"earnings_processed_n{earnings_size}.parquet"
combined_out = PROCESSED_DIR / f"combined_corpus_n{edgar_size + earnings_size}.parquet"


# Convert the metadata dict to a JSON-encoded string before saving. 
# Parquet handles strings fine; we just lose direct queryability of metadata fields (you'd need json.loads to read them back).

import json

def _serialize_metadata_for_parquet(df: pd.DataFrame) -> pd.DataFrame:
    """Convert metadata dicts to JSON strings so parquet can store them safely."""
    df = df.copy()
    df["metadata"] = df["metadata"].apply(json.dumps)
    return df

edgar_processed.pipe(_serialize_metadata_for_parquet).to_parquet(edgar_out, index=False)
earnings_processed.pipe(_serialize_metadata_for_parquet).to_parquet(earnings_out, index=False)
combined.pipe(_serialize_metadata_for_parquet).to_parquet(combined_out, index=False)

print(f"\nSaved to {PROCESSED_DIR}")

Combined corpus: 479 total document units

By source:
source
edgar       319
earnings    160
Name: count, dtype: int64

By corpus_type:
corpus_type
formal            319
conversational    160
Name: count, dtype: int64

Length distribution by source:
          count     mean      std     min      25%      50%      75%       max
source                                                                        
earnings  160.0  28889.0  13746.0  2837.0  19876.0  27194.0  36515.0   79957.0
edgar     319.0  67307.0  54229.0   520.0  27549.0  53594.0  94218.0  420160.0

Saved to d:\General IT\AI-ML-LJMU\final_thesis\code\data\processed


Convert the metadata dict to a JSON-encoded string before saving. 
Parquet handles strings fine; we just lose direct queryability of metadata fields (you'd need json.loads to read them back).

- It's a one-line change
- You keep the conceptual cleanliness of metadata as a unit
- For Phase 3+ work, I'll always be loading these parquets back into pandas where dicts are fine
- The JSON serialization round-trip costs nothing in practice

When loading later: df["metadata"] = df["metadata"].apply(json.loads)


Fix B (cleaner | flatten metadata into top-level columns):
Instead of nesting metadata as a dict, promote each metadata field to its own column. 
Missing fields become None. Arrow handles mixed-type columns badly, but missing columns gracefully.
This means updating prepare_edgar_documents and prepare_earnings_documents to spread metadata into 
the row dict directly, rather than nesting it. Cleaner long-term but requires editing the cleaner module again.

If ever need to filter on metadata fields (e.g., "give me all rows where ticker=AAPL"), 
Fix B would be slightly nicer. But for this thesis, I'll filter on source, subtype, 
and corpus_type — all of which are already top-level columns. Metadata is mostly for traceability, not querying.

<h5> <b>Mean section length is 67,307 chars. Max is 420,160 chars.</b> That max is a single 10-K section over 400 thousand characters long. A 420K-char section will produce ~200 chunks at typical sizes — and that's just one section of one filing.
Also note your min is 520 chars — that's just above the 500-char threshold. Good sanity check: the threshold is doing its job.
Why this matters for Phase 3: chunking strategy and chunk size will dramatically affect total index size. With this length distribution:
<hr>
</h5>

- Average section → ~30 chunks at 2K chars/chunk
- 319 EDGAR docs × ~30 chunks = ~9,500 chunks from EDGAR alone
- Plus earnings → likely ~12,000-15,000 chunks total

That's manageable but not trivial. Sentence-transformer embedding at 768 dims × 15K chunks ≈ 46 MB of embedding storage. Local CPU encoding will take ~10-20 minutes. Worth knowing now.


<h5>Find the longest EDGAR document and look at it</h5>
<hr>
Also, an outlier check worth doing before Phase 3 - that 420K max is suspicious. 
Could be a legitimate massive 10-K (some industrial conglomerates file enormous filings), 
or it could be a duplicate concatenation bug. 

<h5>Quick investigation:</h5>

In [12]:
# Find the longest EDGAR document and look at it
longest = edgar_processed.nlargest(5, "char_length")[["doc_id", "subtype", "char_length"]]
print(longest)

                    doc_id     subtype  char_length
3     edgar_0000_section_8   section_8       420160
131  edgar_0043_section_1A  section_1A       270359
145   edgar_0047_section_8   section_8       259425
118  edgar_0039_section_1A  section_1A       239019
86   edgar_0030_section_1A  section_1A       232310


In [13]:
# Verify we can load the parquet back and decode metadata
test_load = pd.read_parquet(combined_out)
test_load["metadata"] = test_load["metadata"].apply(json.loads)
print(f"Loaded {len(test_load)} rows")
print(f"Sample metadata: {test_load.iloc[0]['metadata']}")

Loaded 479 rows
Sample metadata: {'filename': '103682_2020.htm', 'cik': '103682', 'year': '2020', 'filing_idx': 0}


In [14]:
# Look at the section_8 fields that were dropped (length < 500)

dropped_s8 = edgar_raw[edgar_raw["section_8"].str.len() < 500][["filename", "section_8"]]
print(f"Filings with section_8 < 500 chars: {len(dropped_s8)}")
print(f"\nSample of dropped section_8 content:\n")
for i, row in dropped_s8.head(5).iterrows():
    text = row["section_8"]
    print(f"--- {row['filename']} ({len(text)} chars) ---")
    print(repr(text[:300]))
    print()

Filings with section_8 < 500 chars: 41

Sample of dropped section_8 content:

--- 1688126_2020.htm (90 chars) ---
'Item 8. Financial Statements and Supplementary Data\nSee pages beginning with page.\nItem 9.'

--- 1784851_2020.htm (153 chars) ---
'Item 8.Financial Statements and Supplementary Data\nThis information appears following Item 16 of this Report and is included herein by reference.\nItem 9.'

--- 1091883_2020.htm (206 chars) ---
'Item 8. Financial Statements and Supplementary Data\nOur consolidated financial statements and the related notes thereto are listed in Item 15(a)(1) on the Index to Consolidated Financial Statements.\nItem 9.'

--- 1628063_2020.htm (212 chars) ---
'ITEM 8.\nFINANCIAL STATEMENTS AND SUPPLEMENTARY DATA\nReference is made to the Consolidated Financial Statements and Consolidated Financial Statement Schedule beginning on page for the required information.\nITEM 9.'

--- 1058290_2020.htm (325 chars) ---
'Item 8. Financial Statements and Supplementary Data

### 8 : Save processing summary for your notes:

In [15]:
# Cell 8 — Save processing summary with sample size in filename

# Compute split counts from the actual processed data
genuine_splits = int((earnings_processed["subtype"] == "prepared_remarks").sum())
degenerate_splits = int(
    earnings_processed["metadata"]
    .apply(lambda m: m.get("degenerate_split", False) if isinstance(m, dict) else False)
    .sum()
)
calls_with_marker = genuine_splits + degenerate_splits
calls_no_marker = int(len(earnings_raw)) - calls_with_marker

summary = {
    "edgar": {
        "n_filings_input": int(len(edgar_raw)),
        "n_units_output": int(len(edgar_processed)),
        "units_per_section": edgar_processed["subtype"].value_counts().to_dict(),
        "avg_length_per_section": edgar_processed.groupby("subtype")["char_length"]
            .mean().round(0).to_dict(),
    },
    "earnings": {
        "n_calls_input": int(len(earnings_raw)),
        "n_units_output": int(len(earnings_processed)),
        "units_by_subtype": earnings_processed["subtype"].value_counts().to_dict(),
        "calls_with_marker": calls_with_marker,
        "calls_genuinely_split": genuine_splits,
        "calls_degenerate_split": degenerate_splits,
        "calls_no_marker": calls_no_marker,
        "headers_stripped": int(
            earnings_processed["metadata"]
            .apply(lambda m: m.get("header_stripped", False) if isinstance(m, dict) else False)
            .sum() > 0  # count unique calls, not units
        ),
    },
    "combined": {
        "total_units": int(len(combined)),
        "by_source": combined["source"].value_counts().to_dict(),
    },
}

In [16]:
edgar_size = len(edgar_raw)
earnings_size = len(earnings_raw)
summary_filename = f"phase_02_processing_summary_n{edgar_size}_{earnings_size}.json"

with open(NOTES_DIR / summary_filename, "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

print(f"Saved: {summary_filename}")

{
  "edgar": {
    "n_filings_input": 100,
    "n_units_output": 319,
    "units_per_section": {
      "section_1": 88,
      "section_7": 88,
      "section_1A": 84,
      "section_8": 59
    },
    "avg_length_per_section": {
      "section_1": 48857.0,
      "section_1A": 87837.0,
      "section_7": 47193.0,
      "section_8": 95596.0
    }
  },
  "earnings": {
    "n_calls_input": 100,
    "n_units_output": 160,
    "units_by_subtype": {
      "prepared_remarks": 60,
      "qa": 60,
      "full": 40
    },
    "calls_with_marker": 73,
    "calls_genuinely_split": 60,
    "calls_degenerate_split": 13,
    "calls_no_marker": 27,
    "headers_stripped": 1
  },
  "combined": {
    "total_units": 479,
    "by_source": {
      "edgar": 319,
      "earnings": 160
    }
  }
}
Saved: phase_02_processing_summary_n100_100.json
